## Load Libraries

In [ ]:
import os                     # Work with environment variables and file paths
import requests               # Send HTTP requests 
import glob                   # Find files using wildcard patterns
import subprocess             # Run external commands or shell processes
from dotenv import load_dotenv    # Load environment variables

from openai import OpenAI                                       # For interacting with the OpenAI API

from langchain_openai import ChatOpenAI                         # For interacting with the OpenAI API via langchain
from langchain_ollama import ChatOllama                         # For interacting with the Ollama API via langchain

from langchain_core.messages import SystemMessage, HumanMessage # Create prompts

from langchain_openai import OpenAIEmbeddings                   # Create embeddings using OpenAI models
from langchain_chroma import Chroma                             # Chroma vector database integration for LangChain
from langchain_huggingface import HuggingFaceEmbeddings         # Create embeddings using HuggingFace models

from langchain_community.document_loaders import DirectoryLoader # load many files from a folder
from langchain_community.document_loaders import TextLoader # load text files into LangChain documents
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Splits large documents into smaller chunks for embedding

import gradio as gr # User interface

from IPython.display import Markdown, display  # Display formatted Markdown output in Jupyter notebooks

import re  # Regular expressions for pattern matching in text

## Load Environment Key

In [ ]:
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

# Go up one level to the project folder
project_dir = os.path.dirname(script_dir)
env_path = os.path.join(project_dir, "env_keys", ".env")

# Access the variable
load_dotenv(dotenv_path=env_path)
openai_api_key = os.getenv("OPENAI_API_KEY")
print("API Key loaded:", openai_api_key is not None)

## Ollma Initialize

In [ ]:
subprocess.Popen("ollama serve", shell=True)

In [ ]:
requests.get("http://localhost:11434").content

In [ ]:
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

## Configuration

In [ ]:
model_openai = "gpt-4.1-mini"
model_ollma = "llama3.2"

db_name = "vector_db"

## Character Text Split

In [ ]:
files = glob.glob("knowledge-base/*")

documents = []

for file_path in files:
    doc_type = os.path.basename(file_path)
    loader = DirectoryLoader(path=file_path, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for docs in folder_docs:
        docs.metadata['doc_type'] = doc_type
        documents.append(docs)
        
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

## Make Vector Database and Store

In [ ]:
# Encorder model for vector embeddings
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embedding).delete_collection()

vector_store = Chroma.from_documents(documents=chunks, embedding=embedding, persist_directory=db_name)
print(f"Vector store created with {vector_store._collection.count()} documents")

## LangChain LLM and Retriver

In [ ]:
retriver = vector_store.as_retriever()

In [ ]:
llm = ChatOllama(temperature=0, model=model_ollma)

In [ ]:
retriver.invoke("Who is James")

## Prompt

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

## Calling RAG and LLM

In [ ]:
def answer_question(questions: str, history="NA"):
    docs = retriver.invoke(questions)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke(input=[SystemMessage(content=system_prompt), HumanMessage(content=questions)])
    return response.content

In [ ]:
Markdown(answer_question(questions="Who is Bishop and his title?"))

## Gradio UI

In [ ]:
gr.ChatInterface(answer_question).launch()